In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

import operators as o
import decoding.toric_decoding as to
import decoding.utils as d

In [ ]:
def meas_stab_z(state, i, j):
    # Face operator (Z1 Z2 Z3 Z4)
    relevant_data_bits = [
        state[2*i, j],
        state[2*i+1, j],
        state[2*i+1, (j+1) % dist],
        state[(2*i+2) % (2*dist), j],
        ]

    parity = sum([1 for el in relevant_data_bits if el not in ['I', 'X']]) % 2

    return parity

def meas_stab_x(state, i, j):
    # Vertex operator (X1 X2 X3 X4)
    relevant_data_bits = [
        state[(2*i-1) % (2*dist), j],
        state[2*i, (j-1) % dist],
        state[2*i, j],
        state[2*i+1, j],
        ]
    
    parity = sum([1 for el in relevant_data_bits if el not in ['I', 'Z']]) % 2
    
    return parity

In [ ]:
#Toric code
# initial_state = np.array([
#     ['I', 'I', 'I',],
#  ['I', 'I', 'I',],
#     ['I', 'I', 'I',],
#  ['I', 'I', 'I',],
#     ['I', 'I', 'I',],
#  ['I', 'Z', 'I',],
# ])

dist = 3
px = 0.04
pz = 0.04
pm = 0.04
rounds = 10

z_stabs = np.zeros([dist, dist])
x_stabs = np.zeros([dist, dist])


initial_state = np.array(2 * dist * [dist * ['I']] )
initial_state = o.Operator([2*dist, dist])

matching_graph = to.build_toric_matching_graph(dist, rounds, px, pz, pm)


x_syndrome = []
z_syndrome = []
for _ in range(rounds):
    x_noise = np.where(np.random.random((2*dist, dist)) < px, 'X', 'I')
    z_noise = np.where(np.random.random((2*dist, dist)) < pz, 'Z', 'I')
    o1, o2 = o.Operator([2*dist, dist]), o.Operator([2*dist, dist])
    o1.ops = x_noise
    o2.ops = z_noise
    initial_state = initial_state * o1 * o2

    z_syndrome_t = np.zeros([dist, dist])
    x_syndrome_t = np.zeros([dist, dist])
    for i in range(dist):
        for j in range(dist):
            bit = meas_stab_z(initial_state, i, j)
            z_syndrome_t[i, j] = bit
            bit = meas_stab_x(initial_state, i, j)
            x_syndrome_t[i, j] = bit

    x_syndrome.append(x_syndrome_t)
    z_syndrome.append(z_syndrome_t)

x_detections, z_detections = d.detect_changes(x_syndrome), d.detect_changes(z_syndrome)

matches = d.match_errors(matching_graph, np.array([x_detections, z_detections]))
print('Z\n', x_detections, '\n\nX \n' ,z_detections)
print(matches)

# TODO: log true errors, check EpEt in Stabilizer (form a continuious loop)
